# 01_data_check

이 노트북은 CGV 리뷰 데이터의 **기초 점검용 노트북**이다.  
주요 목적은 다음과 같다.

- 원본 CSV가 정상적으로 불러와지는지 확인
- 컬럼 구조 및 데이터 타입 확인
- 결측치와 중복치 점검
- `clean_reviews.py` 전처리 모듈이 정상 작동하는지 확인
- 전처리 결과(`df_text`, `df_time`)의 기본 상태 점검
- 저장된 품질 점검표와 요약 통계표 확인

이 노트북은 **탐색 및 검증 목적**으로 사용한다.  
즉, 여기서 데이터를 직접 많이 가공하기보다, 전처리 모듈이 잘 동작하는지 확인하고  
후속 노트북(`02_text_eda.ipynb`)에서 본격적인 EDA와 텍스트 분석을 이어갈 수 있도록 준비하는 단계이다.

In [1]:
# 기본 라이브러리 불러오기
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# 출력 옵션 설정
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

## 1. 프로젝트 경로 설정

현재 노트북은 `project/notebooks/` 폴더 안에 위치한다고 가정한다.  
따라서 상위 폴더인 `project/`를 기준 경로로 잡고,  
`src/preprocessing/clean_reviews.py` 모듈을 import 할 수 있도록 경로를 추가한다.

또한 원본 데이터와 전처리 결과 파일의 경로를 함께 설정한다.

In [2]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()

# 현재 노트북이 cgv_review_project/notebooks 안에서 실행된다고 가정
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
# 혹시 프로젝트 루트에서 직접 실행하는 경우도 대비
elif (CURRENT_DIR / "data").exists() and (CURRENT_DIR / "outputs").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. "
        "노트북을 cgv_review_project 또는 cgv_review_project/notebooks 기준으로 실행해주세요."
    )

# src 폴더 import 가능하도록 경로 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("CURRENT_DIR :", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

# 주요 경로 설정
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cgv_reviews_checkpoint.csv"

# 최종 분석 시 아래 주석을 풀고 사용
# RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cgv_reviews_final.csv"

PROCESSED_TEXT_PATH = PROJECT_ROOT / "data" / "processed" / "reviews_cleaned_text.csv"
PROCESSED_TIME_PATH = PROJECT_ROOT / "data" / "processed" / "reviews_cleaned_time.csv"

QUALITY_REPORT_PATH = PROJECT_ROOT / "outputs" / "tables" / "quality_report.csv"
SUMMARY_STATS_PATH = PROJECT_ROOT / "outputs" / "tables" / "summary_statistics.csv"
DAILY_TREND_PATH = PROJECT_ROOT / "outputs" / "tables" / "daily_trend.csv"
IMAGE_COMPARE_PATH = PROJECT_ROOT / "outputs" / "tables" / "image_comparison.csv"

print("RAW_DATA_PATH       :", RAW_DATA_PATH)
print("PROCESSED_TEXT_PATH :", PROCESSED_TEXT_PATH)
print("PROCESSED_TIME_PATH :", PROCESSED_TIME_PATH)
print("QUALITY_REPORT_PATH :", QUALITY_REPORT_PATH)
print("SUMMARY_STATS_PATH  :", SUMMARY_STATS_PATH)
print("DAILY_TREND_PATH    :", DAILY_TREND_PATH)
print("IMAGE_COMPARE_PATH  :", IMAGE_COMPARE_PATH)

CURRENT_DIR : C:\Users\USER\Desktop\cgv_review_project\notebooks
PROJECT_ROOT: C:\Users\USER\Desktop\cgv_review_project
RAW_DATA_PATH       : C:\Users\USER\Desktop\cgv_review_project\data\raw\cgv_reviews_checkpoint.csv
PROCESSED_TEXT_PATH : C:\Users\USER\Desktop\cgv_review_project\data\processed\reviews_cleaned_text.csv
PROCESSED_TIME_PATH : C:\Users\USER\Desktop\cgv_review_project\data\processed\reviews_cleaned_time.csv
QUALITY_REPORT_PATH : C:\Users\USER\Desktop\cgv_review_project\outputs\tables\quality_report.csv
SUMMARY_STATS_PATH  : C:\Users\USER\Desktop\cgv_review_project\outputs\tables\summary_statistics.csv
DAILY_TREND_PATH    : C:\Users\USER\Desktop\cgv_review_project\outputs\tables\daily_trend.csv
IMAGE_COMPARE_PATH  : C:\Users\USER\Desktop\cgv_review_project\outputs\tables\image_comparison.csv


## 2. 전처리 모듈 불러오기

`clean_reviews.py`에서 작성한 함수와 설정 클래스를 불러온다.

이 노트북에서는 전처리 코드를 다시 작성하지 않고,  
이미 저장해둔 재사용 가능한 모듈을 호출해 사용하는 방식을 취한다.  
이렇게 하면 나중에 Streamlit 앱이나 다른 노트북에서도 동일한 로직을 재사용할 수 있다.

In [3]:
from src.preprocessing.clean_reviews import (
    ReviewPreprocessConfig,
    load_reviews_csv,
    build_quality_report,
    preprocess_reviews,
    build_summary_statistics,
    build_daily_trend,
    build_image_comparison,
)

## 3. 원본 CSV 불러오기

현재는 체크포인트 파일인 `cgv_reviews_checkpoint.csv`를 사용한다.  
추후 전체 수집이 완료되면 `cgv_reviews_final.csv`로 경로만 바꿔서 같은 코드를 사용할 수 있다.

이 단계에서는 원본 데이터의 크기와 컬럼 구성을 우선 확인한다.

In [4]:
# 원본 데이터 불러오기
df_raw = load_reviews_csv(RAW_DATA_PATH)

print(f"원본 데이터 행 수: {df_raw.shape[0]:,}")
print(f"원본 데이터 열 수: {df_raw.shape[1]:,}")

df_raw.head()

원본 데이터 행 수: 46,600
원본 데이터 열 수: 11


,review_id,movie_no,movie_name,author,date,score,like_count,review,movie_kind,has_image,api_start_row
0,38249521,30000927,왕과 사는 남자,영화에도라버린학생,2026-03-30 00:16:23,2,0,기대하고 봤는데 기대한 만큼 좋았음 마지막이 아쉬웠는데\n스토리 특성상 원래 있던 내용을 기본으로 깔고가는거라 \n이해됐음ㅁㅁ,2D 일반,N,0
1,38249497,30000927,왕과 사는 남자,용감한할리퀸28348,2026-03-30 00:03:06,2,0,많이 울고 많이 웃었습니다.\n마음이 너무 짠한영화\n,2D 일반,N,0
2,38249493,30000927,왕과 사는 남자,평화로운뮬란676352,2026-03-30 00:02:07,2,0,특히 마지막 단종의삶을 그대로 표현해낸\n배우들의 연기에 몰입하고 왕의 자리의 힘듬을 잘 그려낸영화입니다\n,2D 일반,N,0
3,38249460,30000927,왕과 사는 남자,행복햄덩이,2026-03-29 23:48:13,2,0,박지훈의 무게감 있는 왕 연기 최고였고 유해진님과의 케미도 너무 좋아서 재밌고 감동깊게 잘 봤습니다\n좋은 작품 보여주셔서 감사합니다,2D 일반,N,0
4,38249448,30000927,왕과 사는 남자,완벽한덤보11733,2026-03-29 23:45:05,2,0,단종과 '엄흥도'의 이야기를 이렇게 풀어냈다는게 참 대단하다는 생각이든다.\n다만 나와 영화의 깊이와 속도가 맞지않아 조금 아쉬웠고 이렇게이렇게 흥행한다고? 싶은 생각도 들었다,2D 일반,N,0


## 4. 컬럼 및 데이터 구조 확인

분석 전에 가장 먼저 확인할 것은 데이터 구조이다.

여기서는 다음을 점검한다.

- 컬럼명
- 데이터 타입
- 표본 몇 행
- 각 컬럼이 어떤 역할을 하는지

특히 `date`, `score`, `like_count`, `review`, `has_image`는  
이후 분석에서 핵심적으로 사용되는 컬럼이므로 우선적으로 점검한다.

In [5]:
# 컬럼 목록 확인
df_raw.columns.tolist()

['review_id',
 'movie_no',
 'movie_name',
 'author',
 'date',
 'score',
 'like_count',
 'review',
 'movie_kind',
 'has_image',
 'api_start_row']

In [6]:
# 데이터 타입과 결측치 개수까지 함께 확인
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 46600 entries, 0 to 46599
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   review_id      46600 non-null  int64
 1   movie_no       46600 non-null  int64
 2   movie_name     46600 non-null  str  
 3   author         46593 non-null  str  
 4   date           46600 non-null  str  
 5   score          46600 non-null  int64
 6   like_count     46600 non-null  int64
 7   review         46600 non-null  str  
 8   movie_kind     46598 non-null  str  
 9   has_image      46600 non-null  str  
 10  api_start_row  46600 non-null  int64
dtypes: int64(5), str(6)
memory usage: 3.9 MB


In [7]:
# 표본 3행 확인
df_raw.sample(3, random_state=42)

,review_id,movie_no,movie_name,author,date,score,like_count,review,movie_kind,has_image,api_start_row
8321,38184710,30000927,왕과 사는 남자,지혜로운미니언즈23132,2026-03-08 16:52:19,2,1,조선시대의 충이 이해되지 않던 사람에게 깊은 충심을 알려준 영화\n자연스러운 통곡에 내 심장의 깊이를 알 수 있었던 영화\n짧은 시간의 표현에도 엄흥도를 사랑한 단종과 그를 사랑한 엄흥도가 자연스럽던 명작,2D 일반,N,8335
38463,38086437,30000927,왕과 사는 남자,시간여행,2026-02-13 00:32:23,2,0,웃다가 울다가 이야기에 푹 빠져 보았습니다.,2D 일반,N,38650
24595,38129100,30000927,왕과 사는 남자,진정한레옹2561,2026-02-21 18:09:44,1,0,배우와 배우연기 '는' 좋았어요\n\n가족과 같이 보기에는 무난해요\n그런데 추천해준사람이 증오스러워요\n그럴 수도 있죠,2D 일반,N,24640


## 5. 원본 데이터 기초 확인

이 단계에서는 원본 데이터 차원에서 다음을 확인한다.

- 영화 번호와 영화명이 단일값인지
- score 값이 예상대로 1, 2만 존재하는지
- has_image 값이 Y, N으로 구성되어 있는지
- 날짜 범위가 어느 정도인지

이는 데이터가 예상한 수집 구조와 일치하는지 확인하기 위한 단계이다.

In [8]:
print("movie_no 고유값 수:", df_raw["movie_no"].nunique())
print("movie_name 고유값 수:", df_raw["movie_name"].nunique())
print("movie_kind 고유값 수:", df_raw["movie_kind"].nunique())

print("\nmovie_no 고유값:", df_raw["movie_no"].unique()[:10])
print("movie_name 고유값:", df_raw["movie_name"].unique()[:10])
print("movie_kind 고유값:", df_raw["movie_kind"].unique()[:10])

movie_no 고유값 수: 1
movie_name 고유값 수: 1
movie_kind 고유값 수: 1

movie_no 고유값: [30000927]
movie_name 고유값: <StringArray>
['왕과 사는 남자']
Length: 1, dtype: str
movie_kind 고유값: <StringArray>
['2D 일반', nan]
Length: 2, dtype: str


In [9]:
print("score 분포")
display(df_raw["score"].value_counts(dropna=False).sort_index())

print("\nhas_image 분포")
display(df_raw["has_image"].value_counts(dropna=False))

score 분포


score
1     1334
2    45266
Name: count, dtype: int64


has_image 분포


has_image
N    46111
Y      489
Name: count, dtype: int64

In [10]:
# 원본 date 컬럼 예시 확인
df_raw["date"].head(10)

0    2026-03-30 00:16:23
1    2026-03-30 00:03:06
2    2026-03-30 00:02:07
3    2026-03-29 23:48:13
4    2026-03-29 23:45:05
5    2026-03-29 23:42:17
6    2026-03-29 23:40:51
7    2026-03-29 23:34:26
8    2026-03-29 23:27:54
9    2026-03-29 23:15:40
Name: date, dtype: str

## 6. 결측치 / 중복치 점검

EDA 이전에 반드시 확인해야 할 기본 항목이다.

여기서는 다음을 확인한다.

- 컬럼별 결측치 개수
- `review_id` 중복 여부
- 빈 문자열 리뷰 존재 여부
- 날짜 파싱 가능 여부

이 단계는 데이터 품질을 빠르게 판단하기 위한 체크 포인트다.

In [11]:
# 컬럼별 결측치 개수
missing_counts = df_raw.isna().sum().sort_values(ascending=False)
missing_counts

author           7
movie_kind       2
review_id        0
movie_name       0
movie_no         0
score            0
date             0
like_count       0
review           0
has_image        0
api_start_row    0
dtype: int64

In [12]:
# review_id 중복 개수
duplicate_review_id_count = df_raw["review_id"].duplicated().sum()
print("review_id 중복 개수:", duplicate_review_id_count)

review_id 중복 개수: 0


In [13]:
# 빈 리뷰(공백 포함) 개수 확인
blank_review_mask = df_raw["review"].fillna("").astype(str).str.strip().eq("")
print("빈 리뷰 개수:", blank_review_mask.sum())

빈 리뷰 개수: 1


In [14]:
# 날짜 파싱 가능 여부 확인
date_parsed = pd.to_datetime(df_raw["date"], errors="coerce")
print("날짜 파싱 실패 개수:", date_parsed.isna().sum())

print("최소 날짜:", date_parsed.min())
print("최대 날짜:", date_parsed.max())

날짜 파싱 실패 개수: 0
최소 날짜: 2026-02-07 17:01:23
최대 날짜: 2026-03-30 00:16:23


## 7. 품질 점검표 생성

`clean_reviews.py`의 `build_quality_report()` 함수를 사용해  
원본 데이터 기준 품질 점검표를 생성한다.

이 표는 다음 내용을 한 번에 요약해준다.

- 전체 행 수 / 열 수
- 날짜 파싱 실패 개수
- 중복 리뷰 개수
- 빈 리뷰 개수
- 주요 컬럼 결측치 개수

즉, 노트북에서 개별적으로 확인한 내용을 하나의 표로 다시 정리하는 단계이다.

In [15]:
quality_report = build_quality_report(df_raw, dedup_col="review_id")
quality_report

,metric,value
0,row_count,46600
1,column_count,11
2,date_parse_fail_count,0
3,duplicate_review_id_count,0
4,blank_review_count,1
5,missing_review_count,0
6,missing_score_count,0
7,missing_like_count_count,0
8,missing_author_count,7


## 8. 전처리 설정 및 실행

이제 `ReviewPreprocessConfig`를 이용해 전처리 옵션을 설정하고,  
`preprocess_reviews()` 함수를 호출해 실제 전처리를 수행한다.

현재 설정의 핵심은 다음과 같다.

- 체크포인트 파일 사용
- 부분 수집일 `2026-03-30` 제외 옵션 활성화
- 빈 리뷰 제거
- `review_id` 기준 중복 제거

전처리 결과는 두 개의 데이터프레임으로 나뉜다.

- `df_text` : 전체 텍스트 분석용 데이터  
  - 부분 수집일 포함
- `df_time` : 날짜별 추이 분석용 데이터  
  - 부분 수집일 제외 가능

In [16]:
config = ReviewPreprocessConfig(
    csv_path=str(RAW_DATA_PATH),
    exclude_partial_dates=True,
    partial_dates=("2026-03-30",),
    drop_empty_reviews=True,
    dedup_col="review_id",
)

df_text, df_time = preprocess_reviews(df_raw, config=config)

print("df_text 행 수:", f"{len(df_text):,}")
print("df_time 행 수:", f"{len(df_time):,}")

df_text 행 수: 46,599
df_time 행 수: 46,596


## 9. 전처리 결과 기본 확인

전처리 후 생성된 데이터에는 여러 파생 변수가 추가된다.

예를 들면 다음과 같다.

- `dt` : datetime 형태의 날짜
- `review_date` : 날짜만 추출한 값
- `review_hour` : 리뷰 작성 시각
- `sentiment_label` : 점수 기반 라벨
- `review_length` : 리뷰 길이
- `word_count` : 공백 기준 단어 수
- `has_image_flag` : 이미지 포함 여부 이진화
- `is_partial_day` : 부분 수집일 여부
- `like_count_within_date_rank_pct` : 같은 날짜 안에서 좋아요 상대 위치

이 단계에서는 전처리 결과에 원하는 컬럼이 제대로 생성되었는지 확인한다.

In [17]:
df_text.head()

,review_id,movie_no,movie_name,author,date,score,like_count,review,movie_kind,has_image,api_start_row,dt,review_date,review_year,review_month,review_day,review_hour,review_weekday,sentiment_label,is_positive,is_negative,has_image_flag,review_length,word_count,like_bin,is_partial_day,like_count_within_date_rank_pct
0,38249521,30000927,왕과 사는 남자,영화에도라버린학생,2026-03-30 00:16:23,2,0,기대하고 봤는데 기대한 만큼 좋았음 마지막이 아쉬웠는데\n스토리 특성상 원래 있던 내용을 기본으로 깔고가는거라 \n이해됐음ㅁㅁ,2D 일반,N,0,2026-03-30 00:16:23,2026-03-30,2026,3,30,0,Monday,좋았어요,1,0,0,68,15,0,1,0.666667
1,38249497,30000927,왕과 사는 남자,용감한할리퀸28348,2026-03-30 00:03:06,2,0,많이 울고 많이 웃었습니다.\n마음이 너무 짠한영화,2D 일반,N,0,2026-03-30 00:03:06,2026-03-30,2026,3,30,0,Monday,좋았어요,1,0,0,27,7,0,1,0.666667
2,38249493,30000927,왕과 사는 남자,평화로운뮬란676352,2026-03-30 00:02:07,2,0,특히 마지막 단종의삶을 그대로 표현해낸\n배우들의 연기에 몰입하고 왕의 자리의 힘듬을 잘 그려낸영화입니다,2D 일반,N,0,2026-03-30 00:02:07,2026-03-30,2026,3,30,0,Monday,좋았어요,1,0,0,63,13,0,1,0.666667
3,38249460,30000927,왕과 사는 남자,행복햄덩이,2026-03-29 23:48:13,2,0,박지훈의 무게감 있는 왕 연기 최고였고 유해진님과의 케미도 너무 좋아서 재밌고 감동깊게 잘 봤습니다\n좋은 작품 보여주셔서 감사합니다,2D 일반,N,0,2026-03-29 23:48:13,2026-03-29,2026,3,29,23,Sunday,좋았어요,1,0,0,73,18,0,0,0.116788
4,38249448,30000927,왕과 사는 남자,완벽한덤보11733,2026-03-29 23:45:05,2,0,단종과 '엄흥도'의 이야기를 이렇게 풀어냈다는게 참 대단하다는 생각이든다.\n다만 나와 영화의 깊이와 속도가 맞지않아 조금 아쉬웠고 이렇게이렇게 흥행한다고? 싶은 생각도 들었다,2D 일반,N,0,2026-03-29 23:45:05,2026-03-29,2026,3,29,23,Sunday,좋았어요,1,0,0,97,21,0,0,0.116788


In [18]:
df_text.columns.tolist()

['review_id',
 'movie_no',
 'movie_name',
 'author',
 'date',
 'score',
 'like_count',
 'review',
 'movie_kind',
 'has_image',
 'api_start_row',
 'dt',
 'review_date',
 'review_year',
 'review_month',
 'review_day',
 'review_hour',
 'review_weekday',
 'sentiment_label',
 'is_positive',
 'is_negative',
 'has_image_flag',
 'review_length',
 'word_count',
 'like_bin',
 'is_partial_day',
 'like_count_within_date_rank_pct']

In [19]:
df_text[[
    "review_id",
    "date",
    "dt",
    "review_date",
    "review_hour",
    "score",
    "sentiment_label",
    "like_count",
    "like_count_within_date_rank_pct",
    "has_image",
    "has_image_flag",
    "review_length",
    "word_count",
    "is_partial_day",
]].head(10)

,review_id,date,dt,review_date,review_hour,score,sentiment_label,like_count,like_count_within_date_rank_pct,has_image,has_image_flag,review_length,word_count,is_partial_day
0,38249521,2026-03-30 00:16:23,2026-03-30 00:16:23,2026-03-30,0,2,좋았어요,0,0.666667,N,0,68,15,1
1,38249497,2026-03-30 00:03:06,2026-03-30 00:03:06,2026-03-30,0,2,좋았어요,0,0.666667,N,0,27,7,1
2,38249493,2026-03-30 00:02:07,2026-03-30 00:02:07,2026-03-30,0,2,좋았어요,0,0.666667,N,0,63,13,1
3,38249460,2026-03-29 23:48:13,2026-03-29 23:48:13,2026-03-29,23,2,좋았어요,0,0.116788,N,0,73,18,0
4,38249448,2026-03-29 23:45:05,2026-03-29 23:45:05,2026-03-29,23,2,좋았어요,0,0.116788,N,0,97,21,0
5,38249441,2026-03-29 23:42:17,2026-03-29 23:42:17,2026-03-29,23,2,좋았어요,0,0.116788,N,0,23,7,0
6,38249436,2026-03-29 23:40:51,2026-03-29 23:40:51,2026-03-29,23,2,좋았어요,1,0.534672,N,0,37,9,0
7,38249431,2026-03-29 23:34:26,2026-03-29 23:34:26,2026-03-29,23,2,좋았어요,2,0.910584,N,0,96,26,0
8,38249416,2026-03-29 23:27:54,2026-03-29 23:27:54,2026-03-29,23,2,좋았어요,0,0.116788,N,0,14,2,0
9,38249382,2026-03-29 23:15:40,2026-03-29 23:15:40,2026-03-29,23,2,좋았어요,1,0.534672,N,0,104,22,0


## 10. 전처리 후 데이터 상태 비교

전처리 전후의 크기를 비교하면  
실제로 제거된 데이터가 무엇인지 대략적으로 파악할 수 있다.

예를 들어 아래를 비교할 수 있다.

- 원본 행 수
- 전처리 후 전체 텍스트 분석용 행 수
- 전처리 후 시계열 분석용 행 수
- 부분 수집일 제외 여부에 따른 차이

In [20]:
comparison_df = pd.DataFrame({
    "dataset": ["df_raw", "df_text", "df_time"],
    "row_count": [len(df_raw), len(df_text), len(df_time)],
})

comparison_df

,dataset,row_count
0,df_raw,46600
1,df_text,46599
2,df_time,46596


In [21]:
# 부분 수집일 데이터 개수 확인
partial_day_rows = df_text[df_text["review_date"] == "2026-03-30"]
print("부분 수집일(2026-03-30) 행 수:", len(partial_day_rows))

partial_day_rows[["review_id", "date", "score", "like_count", "review"]].head(10)

부분 수집일(2026-03-30) 행 수: 3


,review_id,date,score,like_count,review
0,38249521,2026-03-30 00:16:23,2,0,기대하고 봤는데 기대한 만큼 좋았음 마지막이 아쉬웠는데\n스토리 특성상 원래 있던 내용을 기본으로 깔고가는거라 \n이해됐음ㅁㅁ
1,38249497,2026-03-30 00:03:06,2,0,많이 울고 많이 웃었습니다.\n마음이 너무 짠한영화
2,38249493,2026-03-30 00:02:07,2,0,특히 마지막 단종의삶을 그대로 표현해낸\n배우들의 연기에 몰입하고 왕의 자리의 힘듬을 잘 그려낸영화입니다


## 11. 기초 통계 요약표 생성

`build_summary_statistics()` 함수를 사용해  
전처리된 전체 데이터(`df_text`) 기준 기초 통계표를 만든다.

이 표에는 다음이 포함된다.

- 전체 리뷰 수
- 긍정 / 부정 리뷰 수
- 긍정 / 부정 비율
- 좋아요 평균 / 중앙값
- 리뷰 길이 평균 / 중앙값
- 이미지 리뷰 수 / 비율
- 고유 작성자 수
- 날짜 범위

이는 전체 데이터셋의 기본 성격을 한눈에 보는 용도이다.

In [22]:
summary_stats = build_summary_statistics(df_text)
summary_stats

,metric,value
0,total_reviews,46599
1,positive_reviews,45265
2,negative_reviews,1334
3,positive_ratio,0.9714
4,negative_ratio,0.0286
5,avg_like_count,0.3556
6,median_like_count,0.0
7,avg_review_length,42.0255
8,median_review_length,31.0
9,image_review_count,489


## 12. 날짜별 추이 테이블 생성

시계열 분석용 데이터(`df_time`)를 기준으로  
날짜별 리뷰 수와 간단한 요약값을 집계한다.

여기서는 다음과 같은 컬럼이 생성된다.

- `review_count`
- `positive_ratio`
- `avg_like_count`
- `median_like_count`
- `avg_review_length`
- `image_ratio`

중요한 점은 이 테이블은 **부분 수집일 제외 옵션이 반영된 데이터**를 기준으로 만든다는 것이다.  
즉, 날짜별 추이 왜곡을 줄이기 위한 목적이다.

In [23]:
daily_trend = build_daily_trend(df_time)
daily_trend.head()

,review_date,review_count,positive_ratio,avg_like_count,median_like_count,avg_review_length,image_ratio
0,2026-02-07,1195,0.975732,0.235146,0.0,43.514644,0.007531
1,2026-02-08,2395,0.980376,0.309395,0.0,43.148225,0.009603
2,2026-02-09,1315,0.968061,0.318631,0.0,40.619772,0.007605
3,2026-02-10,1129,0.967228,0.396811,0.0,40.311780,0.009743
4,2026-02-11,1091,0.971586,0.390467,0.0,40.234647,0.014665


In [24]:
daily_trend.tail(10)

,review_date,review_count,positive_ratio,avg_like_count,median_like_count,avg_review_length,image_ratio
41,2026-03-20,305,0.963934,0.970492,1.0,43.540984,0.006557
42,2026-03-21,404,0.965347,0.594059,0.0,41.992574,0.014851
43,2026-03-22,499,0.959920,1.088176,1.0,38.244489,0.016032
44,2026-03-23,278,0.924460,1.017986,1.0,38.568345,0.017986
45,2026-03-24,168,0.904762,1.345238,1.0,45.666667,0.035714
46,2026-03-25,294,0.955782,0.438776,0.0,40.772109,0.003401
47,2026-03-26,206,0.922330,1.048544,1.0,40.252427,0.009709
48,2026-03-27,183,0.956284,1.120219,1.0,40.060109,0.005464
49,2026-03-28,279,0.974910,1.129032,1.0,40.569892,0.010753
50,2026-03-29,274,0.959854,0.952555,1.0,42.791971,0.007299


## 13. 이미지 포함 여부 비교표 생성

이미지 포함 리뷰(`Y`)와 이미지 미포함 리뷰(`N`)를 비교하는 표를 생성한다.

이 비교는 exploratory 수준에서 의미가 있다.  
표본 수 차이가 크기 때문에 해석은 조심해야 하지만,  
다음과 같은 질문에 답할 수 있다.

- 이미지 포함 리뷰가 더 긍정적인가?
- 이미지 포함 리뷰가 더 긴가?
- 이미지 포함 리뷰가 더 많은 좋아요를 받는가?

이 단계에서는 우선 기초 비교표만 확인한다.

In [25]:
image_compare = build_image_comparison(df_text)
image_compare

,has_image,review_count,positive_ratio,avg_like_count,median_like_count,avg_review_length,median_review_length
0,N,46110,0.971134,0.355086,0.0,41.829234,31.0
1,Y,489,0.993865,0.400818,0.0,60.527607,55.0


## 14. 저장된 CSV 결과 다시 불러오기

앞서 `clean_reviews.py`를 실행하면서 이미 생성된 결과 파일이 있다.  
이 단계에서는 저장된 CSV 파일을 다시 읽어와서,  
모듈 실행 결과와 저장 파일이 일관적인지 확인한다.

즉, 코드 실행 결과와 파일 저장 결과가 동일한 흐름으로 이어졌는지 검증하는 단계이다.

In [26]:
quality_report_saved = pd.read_csv(QUALITY_REPORT_PATH)
summary_stats_saved = pd.read_csv(SUMMARY_STATS_PATH)
daily_trend_saved = pd.read_csv(DAILY_TREND_PATH)
image_compare_saved = pd.read_csv(IMAGE_COMPARE_PATH)

print("quality_report_saved:", quality_report_saved.shape)
print("summary_stats_saved:", summary_stats_saved.shape)
print("daily_trend_saved:", daily_trend_saved.shape)
print("image_compare_saved:", image_compare_saved.shape)

quality_report_saved: (9, 2)
summary_stats_saved: (14, 2)
daily_trend_saved: (51, 7)
image_compare_saved: (2, 7)


In [27]:
quality_report_saved

,metric,value
0,row_count,46600
1,column_count,11
2,date_parse_fail_count,0
3,duplicate_review_id_count,0
4,blank_review_count,1
5,missing_review_count,0
6,missing_score_count,0
7,missing_like_count_count,0
8,missing_author_count,7


In [28]:
summary_stats_saved

,metric,value
0,total_reviews,46599
1,positive_reviews,45265
2,negative_reviews,1334
3,positive_ratio,0.9714
4,negative_ratio,0.0286
5,avg_like_count,0.3556
6,median_like_count,0.0
7,avg_review_length,42.0255
8,median_review_length,31.0
9,image_review_count,489


In [29]:
daily_trend_saved.head()

,review_date,review_count,positive_ratio,avg_like_count,median_like_count,avg_review_length,image_ratio
0,2026-02-07,1195,0.975732,0.235146,0.0,43.514644,0.007531
1,2026-02-08,2395,0.980376,0.309395,0.0,43.148225,0.009603
2,2026-02-09,1315,0.968061,0.318631,0.0,40.619772,0.007605
3,2026-02-10,1129,0.967228,0.396811,0.0,40.311780,0.009743
4,2026-02-11,1091,0.971586,0.390467,0.0,40.234647,0.014665


In [30]:
image_compare_saved

,has_image,review_count,positive_ratio,avg_like_count,median_like_count,avg_review_length,median_review_length
0,N,46110,0.971134,0.355086,0.0,41.829234,31.0
1,Y,489,0.993865,0.400818,0.0,60.527607,55.0


## 15. 전처리 결과 파일 존재 여부 확인

최종적으로 실제 파일들이 원하는 위치에 잘 생성되었는지 확인한다.

이 단계는 단순하지만 중요하다.  
왜냐하면 후속 노트북(`02_text_eda.ipynb`)이나 향후 Streamlit 앱에서  
이 파일들을 다시 불러와 사용할 가능성이 높기 때문이다.

In [31]:
files_to_check = [
    RAW_DATA_PATH,
    PROCESSED_TEXT_PATH,
    PROCESSED_TIME_PATH,
    QUALITY_REPORT_PATH,
    SUMMARY_STATS_PATH,
    DAILY_TREND_PATH,
    IMAGE_COMPARE_PATH,
]

file_check_df = pd.DataFrame({
    "path": [str(p) for p in files_to_check],
    "exists": [p.exists() for p in files_to_check],
})

file_check_df

,path,exists
0,C:\Users\USER\Desktop\cgv_review_project\data\raw\cgv_reviews_checkpoint.csv,True
1,C:\Users\USER\Desktop\cgv_review_project\data\processed\reviews_cleaned_text.csv,True
2,C:\Users\USER\Desktop\cgv_review_project\data\processed\reviews_cleaned_time.csv,True
3,C:\Users\USER\Desktop\cgv_review_project\outputs\tables\quality_report.csv,True
4,C:\Users\USER\Desktop\cgv_review_project\outputs\tables\summary_statistics.csv,True
5,C:\Users\USER\Desktop\cgv_review_project\outputs\tables\daily_trend.csv,True
6,C:\Users\USER\Desktop\cgv_review_project\outputs\tables\image_comparison.csv,True


## 16. 데이터 체크 단계 정리

이 노트북에서 확인한 내용은 다음과 같다.

- 원본 CSV가 정상적으로 불러와짐
- 컬럼 구조와 데이터 타입 확인 완료
- 결측치 / 중복치 / 빈 리뷰 점검 완료
- 날짜 파싱 가능 여부 확인 완료
- 전처리 모듈 실행 확인 완료
- `df_text`, `df_time` 생성 확인 완료
- 품질 점검표 / 요약 통계표 / 날짜별 추이표 / 이미지 비교표 생성 확인 완료

이제 다음 단계에서는 `02_text_eda.ipynb`에서 본격적으로 다음을 수행할 수 있다.

- 리뷰 수 집계 및 시각화
- 날짜별 추이 시각화
- 리뷰 길이 분석
- 단어 빈도
- TF-IDF
- n-gram
- 샘플 기반 토픽 탐색